In [1]:
import pandas as pd
import numpy as np
import os
import requests
import time
from pathlib import Path

In [2]:
!pip install openpyxl -q

In [4]:
import pandas as pd

df_clean = pd.read_excel(
    "df_clean.xlsx",
    engine="openpyxl"
)

print(df_clean.shape)

df_clean.head()

(180519, 43)


,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Profit Per Order,Order Region,Order State,Order Status,Product Card Id,Product Category Id,Product Name,Product Price,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,91.250000,Southeast Asia,Java Occidental,COMPLETE,1360,73,Smart watch,327.75,2018-02-03 22:56:00,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,-249.089996,South Asia,Rajastán,PENDING,1360,73,Smart watch,327.75,2018-01-18 12:27:00,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,-247.779999,South Asia,Rajastán,CLOSED,1360,73,Smart watch,327.75,2018-01-17 12:06:00,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,...,22.860001,Oceania,Queensland,COMPLETE,1360,73,Smart watch,327.75,2018-01-16 11:45:00,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,...,134.210007,Oceania,Queensland,PENDING_PAYMENT,1360,73,Smart watch,327.75,2018-01-15 11:24:00,Standard Class


In [5]:
df_clean = df_clean.sort_values(
    ["order date (DateOrders)", "Order Item Id"]
).copy()

In [6]:
test_start_date = pd.Timestamp("2017-06-14 07:19:00")

In [7]:
df_history = df_clean[
    df_clean["order date (DateOrders)"] < test_start_date
].copy()

df_test_tool = df_clean[
    df_clean["order date (DateOrders)"] >= test_start_date
].copy()

In [8]:
def calculate_historical_rates(history, group_col):

    rates = (
        history.groupby(group_col)["Late_delivery_risk"]
        .mean()
    )

    return rates


In [9]:
product_rates = calculate_historical_rates(
    df_history,
    "Product Card Id"
)

market_rates = calculate_historical_rates(
    df_history,
    "Market"
)

shipping_mode_rates = calculate_historical_rates(
    df_history,
    "Shipping Mode"
)


In [10]:
df_test_tool["historical_product_delay_rate"] = (
    df_test_tool["Product Card Id"]
    .map(product_rates)
)

df_test_tool["historical_market_delay_rate"] = (
    df_test_tool["Market"]
    .map(market_rates)
)

df_test_tool["historical_shipping_mode_delay_rate"] = (
    df_test_tool["Shipping Mode"]
    .map(shipping_mode_rates)
)

In [11]:
global_historical_rate = (
    df_history["Late_delivery_risk"].mean()
)

df_test_tool[
    "historical_product_delay_rate"
] = df_test_tool[
    "historical_product_delay_rate"
].fillna(global_historical_rate)

df_test_tool[
    "historical_market_delay_rate"
] = df_test_tool[
    "historical_market_delay_rate"
].fillna(global_historical_rate)

df_test_tool[
    "historical_shipping_mode_delay_rate"
] = df_test_tool[
    "historical_shipping_mode_delay_rate"
].fillna(global_historical_rate)

In [12]:
df_test_tool = df_test_tool.reset_index(drop=True)

In [13]:
print("Historical data rows:", len(df_history))
print("Shipment Tool rows:", len(df_test_tool))

print("\nHistorical global delay rate:")
print(round(global_historical_rate, 4))




Historical data rows: 153439
Shipment Tool rows: 27080

Historical global delay rate:
0.548


In [14]:
print("\nShipment Tool historical rates:")
df_test_tool[
        [
            "historical_product_delay_rate",
            "historical_market_delay_rate",
            "historical_shipping_mode_delay_rate"
        ]
    ].describe()



Shipment Tool historical rates:


,historical_product_delay_rate,historical_market_delay_rate,historical_shipping_mode_delay_rate
count,27080.000000,27080.000000,27080.000000
mean,0.546147,0.553466,0.548596
std,0.021052,0.002501,0.227253
min,0.318182,0.543273,0.381351
25%,0.545276,0.554738,0.381351
50%,0.547964,0.554738,0.381351
75%,0.549561,0.554738,0.766306
max,0.750000,0.554738,0.954173


In [15]:
df_test_tool.columns

Index(['Type', 'Days for shipping (real)', 'Days for shipment (scheduled)',
       'Benefit per order', 'Sales per customer', 'Delivery Status',
       'Late_delivery_risk', 'Category Id', 'Category Name', 'Customer City',
       'Customer Country', 'Customer Id', 'Customer Segment', 'Customer State',
       'Department Id', 'Department Name', 'Latitude', 'Longitude', 'Market',
       'Order City', 'Order Country', 'Order Customer Id',
       'order date (DateOrders)', 'Order Id', 'Order Item Cardprod Id',
       'Order Item Discount', 'Order Item Discount Rate', 'Order Item Id',
       'Order Item Product Price', 'Order Item Profit Ratio',
       'Order Item Quantity', 'Sales', 'Order Item Total',
       'Order Profit Per Order', 'Order Region', 'Order State', 'Order Status',
       'Product Card Id', 'Product Category Id', 'Product Name',
       'Product Price', 'shipping date (DateOrders)', 'Shipping Mode',
       'historical_product_delay_rate', 'historical_market_delay_rate',
    

In [16]:
shipment_columns = [
    "Order Id",
    "Order Item Id",
    "Product Name",
    "Category Name",
    "Department Name",
    "Market",
    "Customer Country",
    "Order Country",
    "Order Region",
    "Order State",
    "Order City",
    "Shipping Mode",
    "Days for shipment (scheduled)",
    "Order Item Quantity",
    "Order Item Product Price",
    "Sales",
    "Order Item Discount",
    "order date (DateOrders)",
    "Latitude",
    "Longitude",

    # Historical risk features
    "historical_product_delay_rate",
    "historical_market_delay_rate",
    "historical_shipping_mode_delay_rate"
]

In [17]:


shipment_tool = df_test_tool[
    shipment_columns
].copy()

In [18]:
rename_map = {
    "Order Id": "order_id",
    "Order Item Id": "shipment_id",
    "Product Name": "product_name",
    "Category Name": "category",
    "Department Name": "department",
    "Market": "market",
    "Customer Country": "customer_country",
    "Order Country": "destination_country",
    "Order Region": "destination_region",
    "Order State": "destination_state",
    "Order City": "destination_city",
    "Shipping Mode": "shipping_mode",
    "Days for shipment (scheduled)": "scheduled_shipping_days",
    "Order Item Quantity": "quantity",
    "Order Item Product Price": "product_price",
    "Sales": "sales",
    "Order Item Discount": "discount",
    "order date (DateOrders)": "order_date",
    "Latitude": "latitude",
    "Longitude": "longitude",
     "historical_product_delay_rate": "historical_product_delay_rate",
    "historical_market_delay_rate": "historical_market_delay_rate",
    "historical_shipping_mode_delay_rate": "historical_shipping_mode_delay_rate"
}

shipment_tool = shipment_tool.rename(
    columns=rename_map
)



In [19]:
shipment_tool.head()

,order_id,shipment_id,product_name,category,department,market,customer_country,destination_country,destination_region,destination_state,...,quantity,product_price,sales,discount,order_date,latitude,longitude,historical_product_delay_rate,historical_market_delay_rate,historical_shipping_mode_delay_rate
0,61333,153440,Merrell Men's All Out Flash Trail Running Sho,Men's Golf Clubs,Outdoors,LATAM,EE. UU.,Brasil,South America,Río de Janeiro,...,1,109.989998,109.989998,6.05,2017-06-14 07:19:00,39.816647,-74.984673,0.482759,0.543273,0.381351
1,61333,153441,Pelican Sunstream 100 Kayak,Water Sports,Fan Shop,LATAM,EE. UU.,Brasil,South America,Río de Janeiro,...,1,199.990005,199.990005,14.00,2017-06-14 07:19:00,39.816647,-74.984673,0.549561,0.543273,0.381351
2,61333,153442,Fitbit The One Wireless Activity & Sleep Trac,Kids' Golf Clubs,Outdoors,LATAM,EE. UU.,Brasil,South America,Río de Janeiro,...,2,99.949997,199.899994,49.98,2017-06-14 07:19:00,39.816647,-74.984673,0.550000,0.543273,0.381351
3,61333,153443,Nike Men's Free 5.0+ Running Shoe,Cardio Equipment,Footwear,LATAM,EE. UU.,Brasil,South America,Río de Janeiro,...,5,99.989998,499.950012,50.00,2017-06-14 07:19:00,39.816647,-74.984673,0.546833,0.543273,0.381351
4,61334,153444,Perfect Fitness Perfect Rip Deck,Cleats,Apparel,LATAM,EE. UU.,Brasil,South America,Santa Catarina,...,1,59.990002,59.990002,7.80,2017-06-14 07:40:00,36.676338,-121.656517,0.549577,0.543273,0.381351


In [20]:
shipment_tool["shipment_reference"] = (
    "SHP-" +
    shipment_tool["shipment_id"].astype(str)
)

In [21]:
shipment_tool.duplicated(subset=["shipment_id"]).sum()

np.int64(0)

In [ ]:
from google.colab import files

# Save as CSV
shipment_tool.to_csv(
    "shipment_tool_data.csv",
    index=False
)

print("CSV saved successfully")

# Download to local machine
files.download("shipment_tool_data.csv")

CSV saved successfully


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [24]:
import numpy as np

# ============================================================
# SUPPLIER INTELLIGENCE DATA
# SYNTHETIC MVP
# ============================================================

np.random.seed(42)

# Categories appearing in TEST data
categories = (
    shipment_tool["category"]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)



In [25]:
# Countries appearing in TEST data
countries = (
    shipment_tool["destination_country"]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)




In [26]:
supplier_names = [
    "Global Supply Partners",
    "Apex Distribution",
    "NorthStar Logistics",
    "Prime Source Trading",
    "Vertex Supply Co",
    "BlueWave Suppliers",
    "Atlas Commerce",
    "Meridian Goods",
    "Pacific Supply Group",
    "EuroTrade Partners",
    "Summit Distribution",
    "Horizon Suppliers",
    "Continental Source",
    "Nova Supply",
    "Enterprise Trade Group",
    "DirectSource Global",
    "Unified Suppliers",
    "Optima Distribution",
    "Pioneer Supply",
    "GlobalLink Trading"
]



In [27]:
supplier_records = []

for i, supplier_name in enumerate(
    supplier_names,
    start=1
):

    supplier_id = f"SUP{i:03d}"

    supplier_country = np.random.choice(
        countries
    )

    # Each supplier supports 3 categories
    supplier_categories = np.random.choice(
        categories,
        size=min(3, len(categories)),
        replace=False
    )

    average_lead_time = round(
        np.random.uniform(2, 10),
        1
    )

    on_time_rate = round(
        np.random.uniform(0.70, 0.98),
        3
    )

    cancellation_rate = round(
        np.random.uniform(0.005, 0.10),
        3
    )

    quality_issue_rate = round(
        np.random.uniform(0.01, 0.08),
        3
    )

    capacity_utilization = round(
        np.random.uniform(0.50, 0.95),
        3
    )

    # Supplier risk score
    risk_score = (
        (1 - on_time_rate) * 0.40 +
        cancellation_rate * 0.20 +
        quality_issue_rate * 0.20 +
        capacity_utilization * 0.20
    )

    risk_score = round(
        risk_score * 100,
        2
    )

    if risk_score < 10:
        risk_level = "LOW"

    elif risk_score < 20:
        risk_level = "MEDIUM"

    else:
        risk_level = "HIGH"

    for category in supplier_categories:

        supplier_records.append({

            "supplier_id":
                supplier_id,

            "supplier_name":
                supplier_name,

            "supplier_country":
                supplier_country,

            "product_category":
                category,

            "average_lead_time_days":
                average_lead_time,

            "on_time_delivery_rate":
                on_time_rate,

            "cancellation_rate":
                cancellation_rate,

            "quality_issue_rate":
                quality_issue_rate,

            "capacity_utilization":
                capacity_utilization,

            "supplier_risk_score":
                risk_score,

            "supplier_risk_level":
                risk_level,

            "data_source":
                "SYNTHETIC_MVP"
        })


supplier_data = pd.DataFrame(
    supplier_records
)


In [28]:
supplier_data.head()

,supplier_id,supplier_name,supplier_country,product_category,average_lead_time_days,on_time_delivery_rate,cancellation_rate,quality_issue_rate,capacity_utilization,supplier_risk_score,supplier_risk_level,data_source
0,SUP001,Global Supply Partners,Papúa Nueva Guinea,Music,9.6,0.970,0.082,0.031,0.544,14.34,MEDIUM,SYNTHETIC_MVP
1,SUP001,Global Supply Partners,Papúa Nueva Guinea,Hockey,9.6,0.970,0.082,0.031,0.544,14.34,MEDIUM,SYNTHETIC_MVP
2,SUP001,Global Supply Partners,Papúa Nueva Guinea,Fitness Accessories,9.6,0.970,0.082,0.031,0.544,14.34,MEDIUM,SYNTHETIC_MVP
3,SUP002,Apex Distribution,China,CDs,3.6,0.899,0.080,0.052,0.917,25.02,HIGH,SYNTHETIC_MVP
4,SUP002,Apex Distribution,China,Men's Clothing,3.6,0.899,0.080,0.052,0.917,25.02,HIGH,SYNTHETIC_MVP


In [ ]:
# Save as CSV
supplier_data.to_csv(
    "supplier_tool_data.csv",
    index=False
)

print("CSV saved successfully")

# Download to local machine
files.download("supplier_tool_data.csv")

CSV saved successfully


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [22]:
# ============================================================
# INVENTORY INTELLIGENCE DATA
# SYNTHETIC MVP
# ============================================================

import pandas as pd
import numpy as np

np.random.seed(42)

# ------------------------------------------------------------
# 1. Get products from the existing Shipment Tool dataset
# ------------------------------------------------------------

product_columns = [
    "product_id",
    "product_name",
    "category",
    "department"
]

inventory_products = (
    shipment_tool[
        [
            "shipment_id",
            "product_name",
            "category",
            "department"
        ]
    ]
    .drop_duplicates(subset=["product_name", "category"])
    .copy()
)

# Create synthetic product IDs
inventory_products["product_id"] = (
    "PROD-" +
    inventory_products.index
    .astype(str)
    .str.zfill(4)
)

inventory_products = inventory_products[
    [
        "product_id",
        "product_name",
        "category",
        "department"
    ]
].reset_index(drop=True)

print("Unique products available for inventory:")
print(len(inventory_products))

inventory_products.head()

Unique products available for inventory:
73


,product_id,product_name,category,department
0,PROD-0000,Merrell Men's All Out Flash Trail Running Sho,Men's Golf Clubs,Outdoors
1,PROD-0001,Pelican Sunstream 100 Kayak,Water Sports,Fan Shop
2,PROD-0002,Fitbit The One Wireless Activity & Sleep Trac,Kids' Golf Clubs,Outdoors
3,PROD-0003,Nike Men's Free 5.0+ Running Shoe,Cardio Equipment,Footwear
4,PROD-0004,Perfect Fitness Perfect Rip Deck,Cleats,Apparel


In [23]:
# ------------------------------------------------------------
# 2. Synthetic warehouse master data
# ------------------------------------------------------------

warehouse_data = [
    ("WH-001", "Los Angeles", "United States"),
    ("WH-002", "Chicago", "United States"),
    ("WH-003", "New York", "United States"),
    ("WH-004", "Dallas", "United States"),
    ("WH-005", "Seattle", "United States"),
    ("WH-006", "Miami", "United States"),
    ("WH-007", "Atlanta", "United States"),
    ("WH-008", "Phoenix", "United States"),
    ("WH-009", "London", "United Kingdom"),
    ("WH-010", "Manchester", "United Kingdom"),
    ("WH-011", "Paris", "France"),
    ("WH-012", "Berlin", "Germany"),
    ("WH-013", "Amsterdam", "Netherlands"),
    ("WH-014", "Madrid", "Spain"),
    ("WH-015", "Singapore", "Singapore"),
    ("WH-016", "Tokyo", "Japan"),
    ("WH-017", "Toronto", "Canada"),
    ("WH-018", "Sydney", "Australia"),
    ("WH-019", "Dubai", "United Arab Emirates"),
    ("WH-020", "São Paulo", "Brazil")
]

warehouses = pd.DataFrame(
    warehouse_data,
    columns=[
        "warehouse_id",
        "warehouse_location",
        "warehouse_country"
    ]
)

print("Number of warehouses:", len(warehouses))

warehouses.head()

Number of warehouses: 20


,warehouse_id,warehouse_location,warehouse_country
0,WH-001,Los Angeles,United States
1,WH-002,Chicago,United States
2,WH-003,New York,United States
3,WH-004,Dallas,United States
4,WH-005,Seattle,United States


In [29]:
# ------------------------------------------------------------
# 3. Generate synthetic inventory records
# ------------------------------------------------------------

inventory_records = []

for _, product in inventory_products.iterrows():

    # Random warehouse
    warehouse = warehouses.sample(
        n=1,
        random_state=np.random.randint(0, 1_000_000)
    ).iloc[0]

    # Random supplier from your existing supplier dataset
    supplier = supplier_data.sample(
        n=1,
        random_state=np.random.randint(0, 1_000_000)
    ).iloc[0]

    # --------------------------------------------------------
    # Demand
    # --------------------------------------------------------

    daily_demand = np.random.randint(10, 150)

    # --------------------------------------------------------
    # Current stock
    #
    # Generate different inventory conditions:
    # LOW / MEDIUM / HEALTHY
    # --------------------------------------------------------

    inventory_condition = np.random.choice(
        [
            "HEALTHY",
            "MEDIUM",
            "CRITICAL"
        ],
        p=[0.55, 0.30, 0.15]
    )

    if inventory_condition == "HEALTHY":

        coverage_target = np.random.uniform(
            10,
            30
        )

    elif inventory_condition == "MEDIUM":

        coverage_target = np.random.uniform(
            5,
            10
        )

    else:

        coverage_target = np.random.uniform(
            1,
            5
        )

    current_stock = int(
        daily_demand * coverage_target
    )

    # --------------------------------------------------------
    # Reserved stock
    # --------------------------------------------------------

    reserved_percentage = np.random.uniform(
        0.05,
        0.30
    )

    reserved_stock = int(
        current_stock * reserved_percentage
    )

    # --------------------------------------------------------
    # Available stock
    # --------------------------------------------------------

    available_stock = (
        current_stock -
        reserved_stock
    )

    # --------------------------------------------------------
    # Safety stock
    #
    # Approximately 2–7 days of demand
    # --------------------------------------------------------

    safety_stock_days = np.random.uniform(
        2,
        7
    )

    safety_stock = int(
        daily_demand *
        safety_stock_days
    )

    # --------------------------------------------------------
    # Supplier lead time
    #
    # Taken from synthetic Supplier Tool
    # --------------------------------------------------------

    supplier_lead_time = float(
        supplier["average_lead_time_days"]
    )

    # --------------------------------------------------------
    # Reorder point
    #
    # Demand during lead time + safety stock
    # --------------------------------------------------------

    reorder_point = int(
        daily_demand *
        supplier_lead_time
        +
        safety_stock
    )

    # --------------------------------------------------------
    # Incoming inventory
    # --------------------------------------------------------

    has_incoming = np.random.choice(
        [True, False],
        p=[0.65, 0.35]
    )

    if has_incoming:

        incoming_quantity = int(
            daily_demand *
            np.random.uniform(5, 20)
        )

        lead_days = int(
            np.ceil(
                supplier_lead_time
                +
                np.random.uniform(-2, 3)
            )
        )

        lead_days = max(
            1,
            lead_days
        )

        expected_replenishment_date = (
            pd.Timestamp("2017-06-14")
            +
            pd.Timedelta(
                days=lead_days
            )
        ).date()

    else:

        incoming_quantity = 0

        expected_replenishment_date = None

    # --------------------------------------------------------
    # Inventory coverage
    # --------------------------------------------------------

    if daily_demand > 0:

        inventory_coverage_days = round(
            available_stock /
            daily_demand,
            2
        )

    else:

        inventory_coverage_days = 999

    # --------------------------------------------------------
    # Stockout risk
    # --------------------------------------------------------

    if available_stock <= safety_stock:

        stockout_risk = "HIGH"

    elif inventory_coverage_days <= 7:

        stockout_risk = "MEDIUM"

    else:

        stockout_risk = "LOW"

    # --------------------------------------------------------
    # Create record
    # --------------------------------------------------------

    inventory_records.append({

        "inventory_id":
            f"INV-{len(inventory_records) + 1:06d}",

        "product_id":
            product["product_id"],

        "product_name":
            product["product_name"],

        "category":
            product["category"],

        "department":
            product["department"],

        "warehouse_id":
            warehouse["warehouse_id"],

        "warehouse_location":
            warehouse["warehouse_location"],

        "warehouse_country":
            warehouse["warehouse_country"],

        "supplier_id":
            supplier["supplier_id"],

        "current_stock":
            current_stock,

        "reserved_stock":
            reserved_stock,

        "available_stock":
            available_stock,

        "daily_demand":
            daily_demand,

        "safety_stock":
            safety_stock,

        "reorder_point":
            reorder_point,

        "incoming_quantity":
            incoming_quantity,

        "expected_replenishment_date":
            expected_replenishment_date,

        "inventory_coverage_days":
            inventory_coverage_days,

        "stockout_risk":
            stockout_risk,

        "data_source":
            "SYNTHETIC_MVP"
    })


inventory_data = pd.DataFrame(
    inventory_records
)

print(
    "Inventory records generated:",
    len(inventory_data)
)

inventory_data.head()

Inventory records generated: 73


,inventory_id,product_id,product_name,category,department,warehouse_id,warehouse_location,warehouse_country,supplier_id,current_stock,reserved_stock,available_stock,daily_demand,safety_stock,reorder_point,incoming_quantity,expected_replenishment_date,inventory_coverage_days,stockout_risk,data_source
0,INV-000001,PROD-0000,Merrell Men's All Out Flash Trail Running Sho,Men's Golf Clubs,Outdoors,WH-004,Dallas,United States,SUP019,401,98,303,47,127,493,738,2017-06-23,6.45,MEDIUM,SYNTHETIC_MVP
1,INV-000002,PROD-0001,Pelican Sunstream 100 Kayak,Water Sports,Fan Shop,WH-005,Seattle,United States,SUP008,826,135,691,47,235,592,615,2017-06-23,14.70,LOW,SYNTHETIC_MVP
2,INV-000003,PROD-0002,Fitbit The One Wireless Activity & Sleep Trac,Kids' Golf Clubs,Outdoors,WH-011,Paris,France,SUP006,3053,584,2469,139,775,1150,2418,2017-06-20,17.76,LOW,SYNTHETIC_MVP
3,INV-000004,PROD-0003,Nike Men's Free 5.0+ Running Shoe,Cardio Equipment,Footwear,WH-012,Berlin,Germany,SUP010,926,81,845,32,157,429,453,2017-06-21,26.41,LOW,SYNTHETIC_MVP
4,INV-000005,PROD-0004,Perfect Fitness Perfect Rip Deck,Cleats,Apparel,WH-015,Singapore,Singapore,SUP013,453,25,428,109,572,888,0,None,3.93,HIGH,SYNTHETIC_MVP


In [30]:
# ============================================================
# INVENTORY DATA QUALITY CHECK
# ============================================================

print("Shape:")
print(inventory_data.shape)

print("\nColumns:")
print(inventory_data.columns.tolist())

print("\nMissing values:")
print(
    inventory_data.isna().sum()
)

print("\nDuplicate inventory IDs:")
print(
    inventory_data["inventory_id"].duplicated().sum()
)

print("\nStockout risk distribution:")
print(
    inventory_data["stockout_risk"]
    .value_counts()
)

print("\nCoverage statistics:")
print(
    inventory_data[
        "inventory_coverage_days"
    ].describe()
)

Shape:
(73, 20)

Columns:
['inventory_id', 'product_id', 'product_name', 'category', 'department', 'warehouse_id', 'warehouse_location', 'warehouse_country', 'supplier_id', 'current_stock', 'reserved_stock', 'available_stock', 'daily_demand', 'safety_stock', 'reorder_point', 'incoming_quantity', 'expected_replenishment_date', 'inventory_coverage_days', 'stockout_risk', 'data_source']

Missing values:
inventory_id                    0
product_id                      0
product_name                    0
category                        0
department                      0
warehouse_id                    0
warehouse_location              0
warehouse_country               0
supplier_id                     0
current_stock                   0
reserved_stock                  0
available_stock                 0
daily_demand                    0
safety_stock                    0
reorder_point                   0
incoming_quantity               0
expected_replenishment_date    25
inventory_coverage

In [31]:
inventory_data["calculated_available_stock"] = (
    inventory_data["current_stock"]
    -
    inventory_data["reserved_stock"]
)

inventory_data["calculated_coverage"] = (
    inventory_data["available_stock"]
    /
    inventory_data["daily_demand"]
)

In [32]:
# ============================================================
# VALIDATE INVENTORY CALCULATIONS
# ============================================================

available_check = (
    inventory_data["available_stock"]
    ==
    (
        inventory_data["current_stock"]
        -
        inventory_data["reserved_stock"]
    )
)

coverage_check = np.isclose(
    inventory_data["inventory_coverage_days"],
    (
        inventory_data["available_stock"]
        /
        inventory_data["daily_demand"]
    ),
    atol=0.01
)

print(
    "Available stock calculations correct:",
    available_check.all()
)

print(
    "Coverage calculations correct:",
    coverage_check.all()
)

Available stock calculations correct: True
Coverage calculations correct: True


In [33]:
# ============================================================
# SAVE INVENTORY TOOL DATA
# ============================================================

inventory_data.to_csv(
    "inventory_tool_data.csv",
    index=False
)

print(
    "Inventory Tool CSV saved successfully."
)

print(
    "Rows:",
    len(inventory_data)
)

print(
    "Columns:",
    len(inventory_data.columns)
)

Inventory Tool CSV saved successfully.
Rows: 73
Columns: 22


In [34]:
from google.colab import files

files.download(
    "inventory_tool_data.csv"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>